<a href="https://colab.research.google.com/github/jeffheaton/app_generative_ai/blob/main/t81_559_class_11_2_mcp_client.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# T81-559: Applications of Generative Artificial Intelligence
**Module 11: Model Context Protocol (MCP)**
* Instructor: [Jeff Heaton](https://sites.wustl.edu/jeffheaton/), McKelvey School of Engineering, [Washington University in St. Louis](https://engineering.wustl.edu/Programs/Pages/default.aspx)
* For more information visit the [class website](https://github.com/jeffheaton/app_generative_ai).

# Module 11 Material

* Part 11.1: Introduction to the Model Context Protocol [[Video]]() [[Notebook]](t81_559_class_11_1_mcp.ipynb)
* **Part 11.2: Using MCP Servers from an Agent** [[Video]]() [[Notebook]](t81_559_class_11_2_mcp_client.ipynb)
* Part 11.3: Building Your Own MCP Server [[Video]]() [[Notebook]](t81_559_class_11_3_mcp_server.ipynb)
* Part 11.4: MCP Resources and Multi-Server Agents [[Video]]() [[Notebook]](t81_559_class_11_4_mcp_multi.ipynb)
* Part 11.5: MCP Security and the Road Ahead [[Video]]() [[Notebook]](t81_559_class_11_5_mcp_security.ipynb)

# Google CoLab Instructions

The following code ensures that Google CoLab is running and maps Google Drive if needed.

In [ ]:
import sys

# MCP launches its servers as subprocesses and hands them this notebook's
# stderr, which must have a real file descriptor. Notebook kernels do not
# provide one, so we point stderr at a log file. The MCP library captures
# stderr the moment it first loads, so this redirect must run before any
# MCP import, which is why it sits at the top of the setup cell.
sys.stderr = open("mcp_server.log", "w")

import os

try:
    from google.colab import drive, userdata
    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# OpenAI Secrets
if COLAB:
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Install needed libraries in CoLab
if COLAB:
    !pip install langchain langchain_openai langchain-mcp-adapters "mcp<2" mcp-server-time mcp-server-fetch

# Part 11.2: Using MCP Servers from an Agent

In this part we connect an agent to MCP servers that other people wrote. We will use two small, official reference servers, installed above as ordinary Python packages: **mcp-server-time**, which answers time and time-zone questions, and **mcp-server-fetch**, which retrieves web pages. Neither has anything to do with LangChain -- they are generic MCP servers, usable from Claude Desktop, Cursor, or any other MCP host. That independence is the whole point.

One practical note on the installation cell: we pin `mcp<2`. In mid-2026 the MCP Python SDK released a 2.0 that the LangChain adapter library has not yet caught up with, so the pin keeps the pair compatible. This is a normal fact of life in a fast-moving ecosystem, and checking such compatibility should become a reflex.

LangChain's bridge to MCP is the `langchain-mcp-adapters` package. Its central class, **MultiServerMCPClient**, takes a dictionary describing each server -- for stdio transport, simply the command that launches it -- and its `get_tools()` method performs the MCP handshake: it starts each server, asks what tools it offers, and wraps each one as a standard LangChain tool.

Note the `await` in the following code: MCP connections are asynchronous, and Colab notebooks support `await` directly in a cell. A second notebook wrinkle was already handled at the top of the setup cell: MCP sends each server's diagnostic output to stderr and requires a stream with a real file descriptor, which notebook kernels do not provide, so the setup cell pointed stderr at the file mcp_server.log. The ordering matters because the MCP library captures stderr the first time it loads; if an MCP cell ever runs before that redirect in a session, restart the runtime and run the setup cell first. When a server misbehaves, mcp_server.log is where its complaints will be.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "time": {
        "transport": "stdio",
        "command": "python",
        "args": ["-m", "mcp_server_time"],
    },
})

tools = await client.get_tools()
for t in tools:
    print(f"{t.name}: {t.description}")

The tool names, descriptions, and parameter schemas you just printed did not come from our code -- they traveled over the protocol from the server. This is the discovery step: the same handshake any MCP host performs.

Because `get_tools()` returns ordinary LangChain tools, they plug directly into the create_agent function from Module 7. The only difference from Module 7 is that MCP tools are asynchronous, so we run the agent with its async methods: `astream` instead of `stream`, `ainvoke` instead of `invoke`.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

MODEL = 'gpt-5.6-luna'

llm = ChatOpenAI(
        model=MODEL,
        use_responses_api=True  # tool calling on gpt-5.6 models requires the Responses API
    )

agent = create_agent(llm, tools)

async for step in agent.astream(
    {"messages": [{"role": "user", "content":
        "What time is it right now in Tokyo, and how many hours ahead of St. Louis is that?"}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

Watch the trace: the agent calls the server's tools -- possibly twice, once for the current time and once for the conversion -- and the answers flow back over stdio from a subprocess it launched itself. The language model never knew MCP was involved; it just saw tools.

## A Second Server: Fetching the Web

Swapping capabilities is now a configuration change, not a programming change. The following connects to the fetch server instead, giving the agent the ability to read pages from the web. One server argument appears in the configuration: the fetch server honors robots.txt by default and our course data site declines robots requests, so we pass --ignore-robots-txt. Fetching your own site this way is fine; leave the protection on when your agent reads sites you do not own.

In [ ]:
client = MultiServerMCPClient({
    "fetch": {
        "transport": "stdio",
        "command": "python",
        "args": ["-m", "mcp_server_fetch", "--ignore-robots-txt"],
    },
})

tools = await client.get_tools()
agent = create_agent(llm, tools)

result = await agent.ainvoke(
    {"messages": [{"role": "user", "content":
        "Fetch https://data.heatonresearch.com/data/t81-559/bios/DD.txt and tell me "
        "the name and job title of the first person mentioned in it."}]}
)
print(result["messages"][-1].content)

The agent just read a file from the course's data site through a generic web-fetch capability that we did not write and did not wire in by hand -- we only named it in a configuration dictionary.

Two servers, one agent API, zero custom integration code. In the next part, you cross to the other side of the protocol and build servers of your own.